# Airbnb Paris price prediction

https://www.kaggle.com/datasets/thedevastator/airbnb-prices-in-european-cities

https://zenodo.org/records/4446043#.Y9Y9ENJBwUE

## Business Scenario

Imagine that a new Airbnb host wants to publish a listing in Paris but does not know what price to charge.

The host can provide basic information about the accommodation, such as the room type, number of bedrooms, guest capacity, and general location indicators.

The goal of the model is to estimate a reasonable price based on similar listings in the dataset.


In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Importing the Data

We will use two datasets:

- `paris_weekdays.csv`: Airbnb listings for weekdays in Paris.
- `paris_weekends.csv`: Airbnb listings for weekends in Paris.

Since we want one single dataset for Paris, we will load both files and then combine them into a single DataFrame.

We will also create a new column called `day_type` to identify whether each row comes from the weekday or weekend dataset.

In [60]:
# Load the Paris Airbnb datasets

paris_weekdays = pd.read_csv("/content/drive/MyDrive/Elogroup/CNH data literacy/Adv/M3/paris_weekdays.csv")
paris_weekends = pd.read_csv("/content/drive/MyDrive/Elogroup/CNH data literacy/Adv/M3/paris_weekends.csv")

# Add a column to identify the type of day

paris_weekdays["day_type"] = "weekday"
paris_weekends["day_type"] = "weekend"

# Combine both datasets into a single DataFrame

df = pd.concat(
    [paris_weekdays, paris_weekends],
    ignore_index=True
)

# Display the first rows

df.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type
0,0,296.159940,Private room,False,True,2.0,True,0,0,10.0,...,1,0.699821,0.193709,518.478947,25.239380,1218.662228,71.608028,2.35385,48.86282,weekday
1,1,288.237487,Private room,False,True,2.0,True,0,0,10.0,...,1,2.100005,0.107221,873.216962,42.507907,1000.543327,58.791463,2.32436,48.85902,weekday
2,2,211.343089,Private room,False,True,2.0,False,0,0,10.0,...,1,3.302325,0.234724,444.556077,21.640840,902.854467,53.051310,2.31714,48.87475,weekday
3,3,298.956100,Entire home/apt,False,False,2.0,False,0,1,9.0,...,1,0.547567,0.195997,542.142014,26.391291,1199.184166,70.463506,2.35600,48.86100,weekday
4,4,247.926181,Entire home/apt,False,False,4.0,False,0,0,7.0,...,1,1.197921,0.103573,406.928958,19.809165,1070.775497,62.918272,2.35915,48.86648,weekday


In [61]:
df.drop("Unnamed: 0", axis=1, inplace=True)

## Data Cleaning

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6688 entries, 0 to 6687
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   realSum                     6688 non-null   float64
 1   room_type                   6688 non-null   object 
 2   room_shared                 6688 non-null   bool   
 3   room_private                6688 non-null   bool   
 4   person_capacity             6688 non-null   float64
 5   host_is_superhost           6688 non-null   bool   
 6   multi                       6688 non-null   int64  
 7   biz                         6688 non-null   int64  
 8   cleanliness_rating          6688 non-null   float64
 9   guest_satisfaction_overall  6688 non-null   float64
 10  bedrooms                    6688 non-null   int64  
 11  dist                        6688 non-null   float64
 12  metro_dist                  6688 non-null   float64
 13  attr_index                  6688 

We don't have any null values.


In [56]:
df[df.duplicated()]

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type


That are no duplicated lines

### Column Selection

- realSum: the full price of accommodation for two people and two nights in EUR
- room_type: the type of the accommodation
- room_shared: dummy variable for shared rooms
- room_private: dummy variable for private rooms
- person_capacity: the maximum number of guests
- host_is_superhost: dummy variable for superhost status
- multi: dummy variable if the listing belongs to hosts with 2-4 offers
- biz: dummy variable if the listing belongs to hosts with more than 4 offers
- cleanliness_rating: cleanliness rating
- guest_satisfaction_overall: overall rating of the listing
- bedrooms: number of bedrooms (0 for studios)
- dist: distance from city centre in km
- metro_dist: distance from nearest metro station in km
- attr_index: attraction index of the listing location
- attr_index_norm: normalised attraction index (0-100)
- rest_index: restaurant index of the listing location
- attr_index_norm: normalised restaurant index (0-100)
- lng: longitude of the listing location
- lat: latitude of the listing location

The target variable is `realSum`, because our goal is to predict the listing price.

The other columns are potential input variables, also called features.

For this project, we will simulate a **new advertiser scenario**.

This means that we only want to use information that a new host would reasonably know before publishing the listing.


We remove `room_shared` and `room_private` because this information is already represented by `room_type`.

We remove `host_is_superhost`, `multi`, and `biz` because they are related to the host profile or previous platform activity.

We remove `cleanliness_rating` and `guest_satisfaction_overall` because a new listing would not have guest reviews yet.

We remove `attr_index` and `rest_index` because we will keep their normalized versions, `attr_index_norm` and `rest_index_norm`.

In [67]:
from joblib.disk import delete_folder
# Define the target variable

target = "realSum"
# Define columns to remove

columns_to_drop = [
    "room_shared",
    "room_private",
    "host_is_superhost",
    "multi",
    "biz",
    "cleanliness_rating",
    "guest_satisfaction_overall",
    "attr_index",
    "rest_index"
]

# Drop the selected columns
df = df.drop(columns=columns_to_drop, errors="ignore")

# Check the remaining columns
df.columns

Index(['realSum', 'room_type', 'person_capacity', 'bedrooms', 'dist',
       'metro_dist', 'attr_index_norm', 'rest_index_norm', 'lng', 'lat',
       'day_type'],
      dtype='object')

## EDA - Exploratory Data Analysis

Let's now do a brief exploratory analysis to understand the data.

In [68]:
df.describe(include="all")

,realSum,room_type,person_capacity,bedrooms,dist,metro_dist,attr_index_norm,rest_index_norm,lng,lat,day_type
count,6688.000000,6688,6688.000000,6688.000000,6688.000000,6688.000000,6688.000000,6688.000000,6688.000000,6688.000000,6688
unique,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,Entire home/apt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,weekend
freq,NaN,5067,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3558
mean,392.531403,NaN,2.953648,0.972787,2.995823,0.227323,18.204358,42.589111,2.343033,48.864343,NaN
std,330.949745,NaN,1.215007,0.642571,1.463542,0.122769,7.759372,15.680438,0.033724,0.017405,NaN
min,92.739305,NaN,2.000000,0.000000,0.071543,0.003220,5.654976,11.933258,2.247880,48.819940,NaN
25%,240.935782,NaN,2.000000,1.000000,1.831552,0.142363,12.764097,30.156870,2.322468,48.852000,NaN
50%,317.597167,NaN,2.000000,1.000000,2.997026,0.207317,16.448810,40.373716,2.346460,48.866080,NaN
75%,462.065430,NaN,4.000000,1.000000,4.050558,0.290475,22.202119,53.072238,2.367052,48.877870,NaN


In [69]:
num_cols = df.select_dtypes(include="number").columns.tolist()

cat_cols = df.select_dtypes(exclude="number").columns.tolist()


print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

Categorical columns: ['room_type', 'day_type']
Numerical columns: ['realSum', 'person_capacity', 'bedrooms', 'dist', 'metro_dist', 'attr_index_norm', 'rest_index_norm', 'lng', 'lat']


Plots for the target variable: realSum, and also for all the numeric and categorical values.

In [73]:
# @title
import plotly.express as px

fig = px.box(
    df,
    x=target,
    title="Price Distribution"
)

fig.show()

Based on this boxplot and the statistic description, there is one clear outlier, probably a mistake, with the price of 16k, that we should remove.

In [74]:
df.drop(df[df[target] >= 16000].index, inplace=True)

In [75]:
# @title
import plotly.express as px

fig = px.box(
    df,
    x=target,
    title="Price Distribution"
)

fig.show()

In [76]:
# @title
import plotly.express as px

fig = px.histogram(
    df,
    x=target,
    title="Price Distribution"
)

fig.show()

We have an assymetrical normal distribution, with some outliers. In this case, we could also set a range of prices to work with and make it easier for the model to predict.

In the tool we will built, we could inform the use of this range, for example.

For now, let's keep it like this.

In [246]:
fig = px.imshow(
    df.corr(numeric_only=True),
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Correlation Heatmap - Numerical Variables"
)

fig.update_layout(
    width=900,
    height=700
)

fig.show()

In [78]:
# @title
import plotly.graph_objects as go
from plotly.subplots import make_subplots

target = "realSum"

# Select numerical columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove the target and coordinates from the analysis dropdown
num_cols = [col for col in num_cols if col not in [target, "lat", "lng"]]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Distribution", "Relationship with Price")
)

for i, col in enumerate(num_cols):

    fig.add_trace(
        go.Histogram(
            x=df[col],
            name="Distribution",
            visible=(i == 0)
        ),
        row=1,
        col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df[col],
            y=df[target],
            mode="markers",
            name="Price relationship",
            visible=(i == 0),
            marker=dict(opacity=0.5)
        ),
        row=1,
        col=2
    )

buttons = []

for i, col in enumerate(num_cols):
    visible = [False] * len(fig.data)
    visible[2 * i] = True
    visible[2 * i + 1] = True

    buttons.append(
        dict(
            label=col,
            method="update",
            args=[
                {"visible": visible},
                {
                    "title.text": f"Numerical Analysis: {col}",
                    "xaxis.title.text": col,
                    "xaxis2.title.text": col,
                    "yaxis2.title.text": "Price"
                }
            ]
        )
    )

fig.update_layout(
    title=f"Numerical Analysis: {num_cols[0]}",
    height=500,
    width=1200,
    updatemenus=[
        dict(
            buttons=buttons,
            x=1.15,
            y=1.2
        )
    ]
)

fig.update_xaxes(title_text=num_cols[0], row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)

fig.update_xaxes(title_text=num_cols[0], row=1, col=2)
fig.update_yaxes(title_text="Price", row=1, col=2)

fig.show()

We can extract interesting information from the numerical variables.

- person_capacity
  - goes from 2 to 6 people.
  - Most are 2, then 4.
  - There is a slight increase in the price for rooms with 4 to 6 people.
- bedrooms
  - goes from 0 to 5 bedrooms.
  - price goes up from 0 to 3, but then goes down on 4 and 5, which is unexpected.
- dist
  - goes from 0 to 8 - continuos values
  - Assymetrical normal distribution
  - No relation to price
- metro_dist
  - goes from 0 to 1.2 - continous values
  - Assymetrical normal distribution
  - From 0 to 0.5 there are higher prices
- attr_index_norm
  - goes from 0 to 100
  - Assymetrical normal distribution
  - From 0 to 40 there are higher prices. Which is weird, because the common sense is higher attractiveness, higher prices. But the baseline, meaning the minimum price for each atrr index goes slightly up.
- rest_index_norm
  - goes from 0 to 100
  - Assymetrical normal distribution
  - No relation to price


In [87]:
# @title
import folium
import pandas as pd
import branca.colormap as cm
# Prepare the data for the map

df_map = df.dropna(subset=["lat", "lng", "realSum"]).copy()

df_map["lat"] = pd.to_numeric(df_map["lat"], errors="coerce")
df_map["lng"] = pd.to_numeric(df_map["lng"], errors="coerce")
df_map["realSum"] = pd.to_numeric(df_map["realSum"], errors="coerce")

df_map = df_map.dropna(subset=["lat", "lng", "realSum"])
# Optional: sample the data if needed
# df_map = df_map.sample(min(1000, len(df_map)), random_state=42)
# Cap the price used for coloring at 2000

price_cap = 2000

df_map["price_for_color"] = df_map["realSum"].clip(upper=price_cap)
# Create the colormap

colormap = cm.LinearColormap(
    colors=["green", "yellow", "orange", "red"],
    vmin=df_map["price_for_color"].min(),
    vmax=price_cap
)

colormap.caption = "Airbnb Price (capped at 2000)"
# Create the map centered on Paris

paris_map = folium.Map(
    location=[df_map["lat"].mean(), df_map["lng"].mean()],
    zoom_start=12,
    tiles="OpenStreetMap"
)
# Add colored markers

for _, row in df_map.iterrows():
    price = row["realSum"]
    color_value = row["price_for_color"]
    color = colormap(color_value)

    popup_text = f"""
    <b>Price:</b> {price:.2f}<br>
    <b>Room type:</b> {row['room_type']}<br>
    <b>Capacity:</b> {row['person_capacity']}<br>
    <b>Bedrooms:</b> {row['bedrooms']}<br>
    <b>Day type:</b> {row['day_type']}
    """

    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=4,
        popup=folium.Popup(popup_text, max_width=250),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        opacity=0.8
    ).add_to(paris_map)
# Add the color legend

colormap.add_to(paris_map)
# Display the map

paris_map

Output hidden; open in https://colab.research.google.com to view.

We can see higher prices - yellows and oranges - more focused near the seine and champs d'elyses. Which make sense, since those are the most important and prestigious areas of the city. There are a couple outliers, for examples the one red dot on the north edge of the city.

In [89]:
# @title
import plotly.graph_objects as go
from plotly.subplots import make_subplots

target = "realSum"

cat_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

cat_cols_plot = [col for col in cat_cols if col != target]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Category Distribution", "Price by Category")
)

for i, col in enumerate(cat_cols_plot):
    visible = i == 0

    counts = df[col].value_counts(dropna=False)

    fig.add_trace(
        go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            name="Distribution",
            visible=visible
        ),
        row=1,
        col=1
    )

    fig.add_trace(
        go.Box(
            x=df[col].astype(str),
            y=df[target],
            name="Price",
            visible=visible
        ),
        row=1,
        col=2
    )

buttons = []

for i, col in enumerate(cat_cols_plot):
    visible = [False] * len(fig.data)

    visible[2 * i] = True
    visible[2 * i + 1] = True

    buttons.append(
        dict(
            label=col,
            method="update",
            args=[
                {"visible": visible},
                {
                    "title.text": f"Categorical Analysis: {col}",
                    "xaxis.title.text": col,
                    "xaxis2.title.text": col,
                    "yaxis.title.text": "Count",
                    "yaxis2.title.text": "Price"
                }
            ]
        )
    )

fig.update_layout(
    title=f"Categorical Analysis: {cat_cols_plot[0]}",
    height=500,
    width=1200,
    barmode="group",
    updatemenus=[
        dict(
            buttons=buttons,
            x=1.2,
            y=1.2
        )
    ]
)

fig.update_xaxes(title_text=cat_cols_plot[0], row=1, col=1)
fig.update_xaxes(title_text=cat_cols_plot[0], row=1, col=2)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Price", row=1, col=2)

fig.show()

- room_type
  - Most are entire home/apt, followed by private room and just a few shared rooms.
  - The shared room have a price rate much smaller then the other two. Then private room with medium prices and entire home/apt with the highest prices, as expected.
- day_type
   - weekend days have 400 more offers
   - don't relate to price

No further cleaning is necessary.

Now we do data transformation for the categorical and numerical variables.

## Train, Validation and Test Split

Before training the models, we need to split the dataset into three parts:

- The **training set** is used to fit the model.
- The **validation set** is used to compare models and tune decisions during development.
- The **test set** is used only at the end to estimate how well the final model performs on unseen data.

Using a separate test set is important because it gives us a more realistic estimate of model performance.

In [91]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[target])
y = df[target]

# Step 1: Train + Temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,           # 30% goes to temp (val + test)
    shuffle=True,
    random_state=42
)

# Step 2: Split temp into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,           # half of temp -> test, half -> validation
    shuffle=True,
    random_state=42
)

# Check sizes - 70% train / 15% valid / 15% test
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

Train: (4680, 10) Validation: (1003, 10) Test: (1004, 10)


## Preprocessing Pipeline

Machine learning models usually require numerical inputs.

In this dataset, we have two types of features:

- **Numerical features**, such as `person_capacity`, `bedrooms`, `dist`, and `metro_dist`.
- **Categorical features**, such as `room_type` and `day_type`.

For numerical features, we will apply **standard scaling** using `StandardScaler`. This transforms the variables so that they have mean 0 and standard deviation 1.

For categorical features, we will apply **one-hot encoding** using `OneHotEncoder`. This converts categories into numerical columns that can be used by machine learning models.

We will use `ColumnTransformer` to apply the correct preprocessing step to each type of column.

In [223]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Remove the target from the feature lists, if it appears there

cat_cols = [col for col in cat_cols if col != target]

num_cols = df.select_dtypes(include="number").columns.tolist()
num_cols = [col for col in num_cols if col != target]

In [224]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [225]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

In [226]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ]
)

## Train the Regression Models

In [227]:
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

import numpy as np
import pandas as pd

### Models

**Linear Regression** is a simple model that tries to predict a numerical value using a straight-line relationship between the features and the target.

**Ridge Regression** is very similar to Linear Regression, but it adds a penalty to avoid very large coefficients. This helps make the model more stable.

**Lasso Regression** is also similar to Linear and Ridge Regression, but it uses a different type of penalty. Lasso can make some coefficients become exactly zero. This means that Lasso can ignore some variables and work as a simple feature selection method.

**Polynomial Regression** is an extension of Linear Regression. It creates new features from the original numerical variables, such as squared terms and interactions. This allows the model to capture curved relationships while still using a linear model.



The other models here work the same as the classification version, but instead of mode, the decision is made by calculating the mean.

In [259]:
models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(random_state=42),
    "KNN Regressor": KNeighborsRegressor(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42)
}

In [260]:
# Preprocessor specifically for Polynomial Regression
from sklearn.preprocessing import PolynomialFeatures

numeric_polynomial_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(
        degree=2,
        include_bias=False
    ))
])

polynomial_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_polynomial_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ]
)

In [261]:
trained_pipelines = {}

for model_name, model in models.items():
    current_preprocessor = preprocessor

    if model_name == "Polynomial Regression":
        current_preprocessor = polynomial_preprocessor


    pipeline = Pipeline(steps=[
        ("preprocessor", current_preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    trained_pipelines[model_name] = pipeline

In [262]:
# Get feature names after preprocessing
feature_names = trained_pipelines["Linear Regression"] \
    .named_steps["preprocessor"] \
    .get_feature_names_out()

# Get coefficients from each linear model
linear_coef = trained_pipelines["Linear Regression"] \
    .named_steps["model"] \
    .coef_

ridge_coef = trained_pipelines["Ridge Regression"] \
    .named_steps["model"] \
    .coef_

lasso_coef = trained_pipelines["Lasso Regression"] \
    .named_steps["model"] \
    .coef_

In [263]:
# Create a comparison table

weights_comparison = pd.DataFrame({
    "Feature": feature_names,
    "Linear Weight": linear_coef,
    "Ridge Weight": ridge_coef,
    "Lasso Weight": lasso_coef
})

weights_comparison

,Feature,Linear Weight,Ridge Weight,Lasso Weight
0,num__person_capacity,81.532678,81.511680,81.538893
1,num__bedrooms,75.027716,75.018383,73.912185
2,num__dist,5.591042,5.529667,-0.000000
3,num__metro_dist,2.717177,2.729026,1.645704
4,num__attr_index_norm,54.905636,54.890993,57.250233
5,num__rest_index_norm,29.903162,29.891769,21.866113
6,num__lng,-26.068861,-26.121718,-28.188578
7,num__lat,-8.194444,-8.211861,-6.531566
8,cat__room_type_Private room,-63.639286,-63.501802,-56.287949
9,cat__room_type_Shared room,-235.443169,-232.042150,-168.030827


For the three types of regression, the weights are very similar. The difference was on num_dist and cat_day_type_weekend that were eliminated on lasso, meaning that it didn't matter much.

### Model Evaluation Summary

The models can be evaluated using MAE, MSE, RMSE, R² and MAPE.

**MAE**, or Mean Absolute Error, shows the average absolute difference between the real price and the predicted price. It is easy to interpret because it is measured in the same unit as the target variable. In this problem, MAE tells us how many euros, on average, the model is missing by.

**MSE**, or Mean Squared Error, measures the average squared difference between the real price and the predicted price. Because the errors are squared, larger mistakes receive much stronger penalties. However, MSE is harder to interpret directly because it is measured in squared price units.

**RMSE**, or Root Mean Squared Error, is the square root of MSE. It brings the error back to the same unit as the target variable, making it easier to interpret than MSE. Like MSE, RMSE penalizes large errors more strongly than MAE.

**R²**, or coefficient of determination, shows how much of the variation in price is explained by the model. A higher R² means the model explains more of the differences between Airbnb prices. An R² close to 1 indicates strong predictive performance, while an R² close to 0 means the model is not much better than predicting the average price.

**MAPE**, or Mean Absolute Percentage Error, shows the average error as a percentage of the real price. This is useful because it gives a business-friendly interpretation. For example, a MAPE of 20% means that, on average, the model predictions are about 20% away from the actual prices.

For this Airbnb price prediction problem, MAE and MAPE are especially useful because they are easy to communicate to a business audience. MSE and RMSE are useful when we want to penalize large pricing errors more strongly. R² helps compare how well different models explain the overall price variation.

In [264]:
results = []

for model_name, model in models.items():

    pipeline = trained_pipelines[model_name]

    y_val_pred = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, y_val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    r2 = r2_score(y_val, y_val_pred)

    results.append({
        "Model": model_name,
        "Validation MAE": mae,
        "Validation RMSE": rmse,
        "Validation R2": r2,
        "Validation MAPE": mean_absolute_percentage_error(y_val, y_val_pred)
    })

In [265]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Validation R2",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Validation MAE,Validation RMSE,Validation R2,Validation MAPE
0,Random Forest Regressor,86.218333,150.706989,0.651106,0.222978
1,Gradient Boosting Regressor,103.050150,166.963155,0.571779,0.271485
2,Polynomial Regression,106.855623,179.816806,0.503308,0.279815
3,KNN Regressor,109.929834,181.785883,0.492371,0.292845
4,Decision Tree Regressor,93.180473,182.405884,0.488902,0.234460
5,Linear Regression,116.669951,194.638422,0.418053,0.318848
6,Ridge Regression,116.671214,194.649834,0.417985,0.318970
7,Lasso Regression,116.632272,195.089429,0.415353,0.321848


## Introducing Cross-Validation

Until now, we trained the models using the training set and evaluated them using one validation set.

This is a good first approach, but it has one limitation: the validation result depends on one specific split of the data.

If the validation set happens to be easier or harder than usual, the result may give us a misleading impression of model performance.

To reduce this problem, we can use **cross-validation**.

In cross-validation, the training data is divided into several parts, called folds. The model is trained multiple times. Each time, it uses some folds for training and one fold for validation.

This allows us to evaluate the model across different validation subsets and obtain a more stable estimate of performance.

The test set is still kept separate and will only be used at the end.

In [266]:
from sklearn.model_selection import cross_validate

# We will apply cross-validation only on the training + validation data.
# The test set remains untouched.

X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

cv_results = []
cv_results = []

for model_name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train_val,
        y_train_val,
        cv=5,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2",
            "mape": "neg_mean_absolute_percentage_error"
        },
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV MAE": -scores["test_mae"].mean(),
        "CV RMSE": -scores["test_rmse"].mean(),
        "CV R2": scores["test_r2"].mean(),
        "CV R2 Std": scores["test_r2"].std(),
        "CV MAPE": -scores["test_mape"].mean(),
        "CV MAPE (%)": -scores["test_mape"].mean() * 100
    })

The problem with cross validation is that it takes k times more time to train - infeasible with larger datasets.

In [267]:
cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="CV R2",
    ascending=False
).reset_index(drop=True)

cv_results_df

,Model,CV MAE,CV RMSE,CV R2,CV R2 Std,CV MAPE,CV MAPE (%)
0,Random Forest Regressor,89.638754,164.304098,0.599691,0.061761,0.234942,23.494155
1,Gradient Boosting Regressor,103.644498,174.021500,0.551642,0.047760,0.275633,27.563350
2,KNN Regressor,112.871001,191.595529,0.457020,0.052300,0.295310,29.530997
3,Ridge Regression,118.405322,197.761389,0.424542,0.052504,0.324946,32.494583
4,Linear Regression,118.416415,197.762536,0.424535,0.052452,0.324960,32.496044
5,Polynomial Regression,118.416415,197.762536,0.424535,0.052452,0.324960,32.496044
6,Lasso Regression,118.133882,197.946962,0.423491,0.052658,0.325287,32.528696
7,Decision Tree Regressor,98.033676,222.391958,0.261778,0.115084,0.249608,24.960754


## Hyperparameters

Until this point, we trained the models using mostly the default hyperparameters from the Scikit-Learn library.

A **hyperparameter** is a configuration value that is chosen before training the model. It is different from a model parameter.

Model parameters are learned from the data. For example, in Linear Regression, the coefficients are learned during training.

Hyperparameters are not learned directly from the data. They control how the model is trained or how complex the model can become.

For example, in a Random Forest, the number of trees is a hyperparameter. In a KNN model, the number of neighbors is a hyperparameter.

Choosing good hyperparameters can strongly affect model performance.

### Examples of Hyperparameters by Model

For each model we tested, there are different hyperparameters that can be adjusted.

**Linear Regression**

Linear Regression has fewer important hyperparameters than other models. In its basic version, we usually use the default configuration from Scikit-Learn.

Some configurations include:

- `fit_intercept`: whether the model should calculate an intercept term.
- `positive`: whether the coefficients should be forced to be positive.

In this notebook, we used the default Linear Regression configuration.

**Ridge Regression**

Ridge Regression is a linear model with regularization. Its most important hyperparameter is:

- `alpha`: controls the strength of regularization.

A larger `alpha` makes the model coefficients smaller and can reduce overfitting.

**Lasso Regression**

Lasso Regression is also a regularized linear model. Its most important hyperparameter is:

- `alpha`: controls the strength of regularization.

Lasso can shrink some coefficients to zero, which means it can also perform a kind of feature selection.

Other useful hyperparameters include:

- `max_iter`: maximum number of iterations used during optimization.
- `tol`: tolerance used to decide when optimization has converged.

**KNN Regressor**

KNN predicts a value based on nearby examples. Important hyperparameters include:

- `n_neighbors`: number of neighbors used to make the prediction.
- `weights`: whether all neighbors have the same importance or closer neighbors have more importance.
- `metric`: distance metric used to compare observations.

**Decision Tree Regressor**

A Decision Tree can easily become too complex if it is not controlled. Important hyperparameters include:

- `max_depth`: maximum depth of the tree.
- `min_samples_split`: minimum number of samples required to split a node.
- `min_samples_leaf`: minimum number of samples required in a leaf.
- `max_features`: number of features considered when looking for the best split.

**Random Forest Regressor**

Random Forest is an ensemble of many decision trees. Important hyperparameters include:

- `n_estimators`: number of trees in the forest.
- `max_depth`: maximum depth of each tree.
- `min_samples_split`: minimum number of samples required to split a node.
- `min_samples_leaf`: minimum number of samples required in a leaf.
- `max_features`: number of features considered by each tree when splitting.
- `bootstrap`: whether each tree is trained on a bootstrap sample of the data.

**Gradient Boosting Regressor**

Gradient Boosting builds trees sequentially, where each new tree tries to correct the errors of the previous ones. Important hyperparameters include:

- `n_estimators`: number of boosting stages.
- `learning_rate`: controls how much each new tree contributes to the final prediction.
- `max_depth`: maximum depth of each individual tree.
- `subsample`: fraction of samples used to fit each tree.
- `min_samples_leaf`: minimum number of samples required in a leaf.

### AutoML with Random Search

So far, we trained each model using manually chosen hyperparameters.

Instead of choosing these values manually, we can use a simple AutoML approach: **Randomized Search**.

Randomized Search tests several random combinations of hyperparameters and selects the best one based on cross-validation performance.

In this section, we will tune a `DecisionTreeRegressor` as an example using `RandomizedSearchCV`.

In [236]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

import numpy as np
# Create the pipeline with preprocessing + Decision Tree Regressor

tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        random_state=42
    ))
])
# Define the hyperparameter search space

param_distributions = {
    "model__max_depth": [2, 3, 5, 7, 10, 15, 20, 30, None],
    "model__min_samples_split": [2, 5, 10, 20, 50],
    "model__min_samples_leaf": [1, 2, 5, 10, 20],
    "model__max_features": [None, "sqrt", "log2"],
    "model__criterion": ["squared_error", "absolute_error"]
}
# Randomized Search with Cross-Validation

random_search = RandomizedSearchCV(
    estimator=tree_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_val, y_train_val)


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['person_capacity',
                                                                                'bedrooms',
                                                                                'dist',
                                                                                'metro_dist',
                                                                                'attr_index_norm',
                                                                                'rest_index_norm',
                                                                                'lng',
                                                                                'lat']),
                                                                              ('cat',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strate...
                                                                                'day_type'])])),
                                             ('model',
                                              DecisionTreeRegressor(random_state=42))]),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'model__criterion': ['squared_error',
                                                             'absolute_error'],
                                        'model__max_depth': [2, 3, 5, 7, 10, 15,
                                                             20, 30, None],
                                        'model__max_features': [None, 'sqrt',
                                                                'log2'],
                                        'model__min_samples_leaf': [1, 2, 5, 10,
                                                                    20],
                                        'model__min_samples_split': [2, 5, 10,
                                                                     20, 50]},
                   random_state=42, scoring='r2')

In [237]:
# Best hyperparameters found
print(random_search.best_params_)

# Best cross-validation score
print(random_search.best_score_)

{'model__min_samples_split': 50, 'model__min_samples_leaf': 10, 'model__max_features': None, 'model__max_depth': 15, 'model__criterion': 'squared_error'}
0.47025728600473615


Even not being as good as Random Forest, this is much better than the previous result for the same model type, which was r2 = 0.26. This shows how relevant hyperparameter optimization is.

In [238]:
trained_pipelines["Decision Tree Regressor"].get_params()

{'memory': None,
 'steps': [('preprocessor',
   ColumnTransformer(transformers=[('num',
                                    Pipeline(steps=[('imputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('scaler', StandardScaler())]),
                                    ['person_capacity', 'bedrooms', 'dist',
                                     'metro_dist', 'attr_index_norm',
                                     'rest_index_norm', 'lng', 'lat']),
                                   ('cat',
                                    Pipeline(steps=[('imputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('encoder',
                                                     OneHotEncoder(drop='first',
                                                                   handle_unknown='ignore'))]),
               

Optimizing hyperparameters is a study area by itself, and is very time consuming and memory consuming.

There are other types of AutoML algorithms for Hyperparameter Optimization

| Method | How it works | Advantages | Limitations |
|---|---|---|---|
| **Manual Search** | The user chooses a few combinations based on intuition or previous experience. | Simple and easy to understand. Useful when starting an experiment. | Depends heavily on human intuition and may miss better combinations. |
| **Grid Search** | Tests every possible combination from a predefined list of values. | Systematic and easy to explain. Guarantees that all combinations in the grid are tested. | Can be very slow when there are many hyperparameters or many possible values. |
| **Random Search** | Randomly samples combinations from a predefined search space. | Usually more efficient than Grid Search when only some hyperparameters strongly affect performance. Can explore a wider range with fewer trials. | Does not guarantee that the best combination in the search space will be tested. |
| **Bayesian Optimization** | Uses previous results to decide which combinations are more promising to test next. | More intelligent than random search. Can find good configurations with fewer trials. | More complex to explain and implement. |
| **Hyperband** | Trains many configurations for a small amount of resources, then keeps only the most promising ones for more training. | Efficient when training is expensive. Avoids spending too much time on bad configurations. | Works best when partial training performance is a good indicator of final performance. |

Extra exercise: Try automl with other algorithms and models.

## Evaluate model

In [239]:
best_model_name = results_df.iloc[0]["Model"]

best_model = trained_pipelines[best_model_name]

print("Best model:", best_model_name)

Best model: Random Forest Regressor


In [240]:
y_test_pred = best_model.predict(X_test)

In [241]:
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

test_results = pd.DataFrame({
    "Model": [best_model_name],
    "Test MAE": [test_mae],
    "Test R2": [test_r2],
    "Test MAPE": [mean_absolute_percentage_error(y_test, y_test_pred)]
})

test_results

,Model,Test MAE,Test R2,Test MAPE
0,Random Forest Regressor,87.977457,0.69227,0.22373


## Feature Importance

In [242]:
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_model.named_steps["model"].feature_importances_
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=True
).reset_index(drop=True)

In [244]:
# @title
fig = px.bar(
    feature_importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    text="Importance",
    title=f"Feature Importances - Random Forest"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)


fig.show()

Interesting to notice that even though person_capacity was the highest corellated, it is only the fourth in importance for the model.

Same applies to rest_index.

Now we could export the pipeline/model and build an application.